# 第18周 · 练习：强化学习的五个零件与一次审计

今天四节课里的算法，拆开来都是几行代码。前五道小题把它们各写一遍，每道题都是**先在纸上算，再写代码，最后用自查对答案**；E6 用讲义 S2 末尾价值迭代的精确答案审计一张 Q 表；E7 是选做，手算一步 Thompson 采样。

| 题 | 内容 | 用在哪里 | 建议时间 |
| --- | --- | --- | --- |
| E1 | 增量均值更新 | S1 老虎机的价值估计 | 10 分钟 |
| E2 | ε-greedy 选动作 | S1、S2、S4 | 15 分钟 |
| E3 | 一步 Bellman 更新；价值如何一格格往回传 | S2、S4 的 Q-learning | 20 分钟 |
| E4 | reward-to-go | S3 的 REINFORCE、作业 | 15 分钟 |
| E5 | 随机打破平局的 argmax | S2 的 FrozenLake 坑 | 15 分钟 |
| E6 | 审计一张 Q 表：贪心策略的精确成功率差在哪几格 | S2 末尾的价值迭代 | 15 分钟 |
| E7（选做） | Beta 后验：臂 0 比臂 1 好的概率 | S1 的 Thompson 采样 | 15 分钟 |

AI 编程助手可以用（第 9 周起的规则），但这几道题要写的代码都只有两三行——先自己写。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：Gymnasium（强化学习环境库；只训练、不渲染画面，所以不需要 pygame）
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium>=1.4,<2"], check=True)
print("Gymnasium 已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from zero2researcher import experiments
from zero2researcher.checks import week18 as ck
from zero2researcher.checks import report

print("gymnasium", gym.__version__)

## E1 · 增量均值更新

某个臂先后拉出的奖励是 $R_1 = 2,\ R_2 = -1,\ R_3 = 3$。估计值初始为 $Q_1 = 0$，按

$$Q_{n+1} = Q_n + \frac{1}{n}\big(R_n - Q_n\big)$$

更新。**E1a（纸上）**：依次算出 $Q_2, Q_3, Q_4$，并验证 $Q_4$ 恰好等于三个奖励的平均。

（在这里写：$Q_2$ = ？，$Q_3$ = ？，$Q_4$ = ？）

**E1b（代码）**：写 `incremental_mean(q, n, r)`：`q` 是前 $n-1$ 个奖励的平均 $Q_n$，`r` 是第 $n$ 个奖励 $R_n$，返回 $Q_{n+1}$。注意 `n` 是**算上这个奖励之后**的次数（第一个奖励到来时 `n = 1`）。

In [ ]:
def incremental_mean(q, n, r):
    # 在这里写你的代码
    raise NotImplementedError


q = 0.0
for n, r in enumerate([2.0, -1.0, 3.0], start=1):
    q = incremental_mean(q, n, r)
    print(f"第 {n} 个奖励 {r:+.0f} 之后，Q = {q:.4f}")
ck.incremental_mean(incremental_mean)

**E1c**：用你的函数处理一长串奖励，确认结果与 `np.mean` 一致——但你只存了两个数（`q` 和 `n`），没有存整串历史：

In [ ]:
rewards = np.random.default_rng(0).normal(1.5, 1.0, size=10_000)
q = 0.0
for n, r in enumerate(rewards, start=1):
    q = incremental_mean(q, n, r)
print(f"增量更新 {q:.6f} | np.mean {rewards.mean():.6f}")

**追问**（一两句）：如果把步长 $1/n$ 换成一个常数 $\alpha = 0.1$，$Q$ 还是样本平均吗？最近的奖励和很久以前的奖励各占多大权重？什么情况下你**宁愿**用常数步长？

（在这里回答）

## E2 · ε-greedy 选动作

**E2a（纸上）**：4 个动作的估计值 $Q = [0.1, 0.5, 0.2, 0.0]$，$\varepsilon = 0.2$。约定：以概率 $\varepsilon$ **从全部 4 个动作里**均匀随机选（所以探索时也可能恰好选到贪心动作），否则选 $\arg\max Q$。动作 1 被选中的概率是多少？其他每个动作呢？

（在这里写：P(动作 1) = ？；P(其他每个动作) = ？）

**E2b（代码）**：写 `epsilon_greedy(Q, epsilon, rng)`，返回动作下标（`int`）。用传进来的 `rng`（`np.random.default_rng` 生成的随机数发生器）抽随机数：`rng.random()` 给一个 [0, 1) 的均匀随机数，`rng.integers(k)` 给 0 … k−1 的随机整数。

In [ ]:
def epsilon_greedy(Q, epsilon, rng):
    # 在这里写你的代码
    raise NotImplementedError


rng = np.random.default_rng(0)
Q_demo = np.array([0.1, 0.5, 0.2, 0.0])
picks = np.array([epsilon_greedy(Q_demo, 0.2, rng) for _ in range(10_000)])
print("10000 次里各动作被选中的频率:", np.round(np.bincount(picks, minlength=4) / len(picks), 3))
ck.epsilon_greedy(epsilon_greedy)

**追问**（一句话）：有些教材的 ε-greedy 在探索时只从**非贪心**动作里选。那样动作 1 的概率是多少？两种约定在 ε 很小时差别大吗？

（在这里回答）

## E3 · 一步 Bellman 更新

一个 3 个状态 × 2 个动作的 Q 表：

| | 动作 0 | 动作 1 |
| --- | --- | --- |
| 状态 0 | 0.0 | 0.5 |
| 状态 1 | 1.0 | 2.0 |
| 状态 2 | −0.5 | 0.25 |

Q-learning 的更新：目标 $= r + \gamma \max_{a'} Q(s', a')$（如果这一步之后回合**终止**，目标 $= r$），新值 $= $ 旧值 $+ \alpha\,($目标 $-$ 旧值$)$。

**E3a（纸上）**，取 $\alpha = 0.5$、$\gamma = 0.9$：

1. 在状态 0 做动作 0，得到 $r = 1$，到达状态 1（未终止）。$Q(0, 0)$ 更新成多少？
2. 在状态 1 做动作 0，得到 $r = 0$，回合终止（掉进了洞）。$Q(1, 0)$ 更新成多少？

（在这里写：1. 目标 = ？，新的 $Q(0,0)$ = ？；2. 目标 = ？，新的 $Q(1,0)$ = ？）

**E3b（代码）**：写 `q_update(Q, s, a, r, s_next, terminated, alpha, gamma)`：**原地**更新 `Q[s, a]`，并返回更新后的值。

In [ ]:
def q_update(Q, s, a, r, s_next, terminated, alpha, gamma):
    # 在这里写你的代码
    raise NotImplementedError


Q_small = np.array([[0.0, 0.5],
                    [1.0, 2.0],
                    [-0.5, 0.25]])
print("1. 新的 Q(0, 0) =", q_update(Q_small.copy(), 0, 0, 1.0, 1, False, 0.5, 0.9))
print("2. 新的 Q(1, 0) =", q_update(Q_small.copy(), 1, 0, 0.0, 2, True, 0.5, 0.9))
ck.bellman_update(q_update)

**E3c · 价值如何一格格往回传**：一条走廊 0 → 1 → 2 → 终点，每个状态只有一个动作"前进"，走到终点得 +1，其余每步 0。取 $\alpha = 1$、$\gamma = 0.9$，让智能体**从头到尾**走同一条路线若干遍，每走一步就用 `q_update` 更新一次。先预测：第 1 遍之后哪几个状态的 Q 不是 0？要走几遍，状态 0 才"知道"终点的存在？

In [ ]:
Q_corridor = np.zeros((3, 1))                       # 3 个状态 × 1 个动作
for sweep in range(1, 5):
    for s in range(3):                               # 按时间顺序：0 → 1 → 2 → 终点
        reached_goal = (s == 2)
        q_update(Q_corridor, s, 0, 1.0 if reached_goal else 0.0, min(s + 1, 2), reached_goal, alpha=1.0, gamma=0.9)
    print(f"第 {sweep} 遍之后 Q = {Q_corridor[:, 0].round(3)}")

**问题**（每题一两句）：

1. 为什么价值每走一遍只往回传一格？如果每一遍都**倒着**更新（先更新状态 2，再 1，再 0），要几遍？
2. 最后的 Q 为什么是 $[0.81, 0.9, 1]$ 而不是 $[1, 1, 1]$？这和 S2 里 FrozenLake 的 Q 表图有什么关系？

（在这里回答）

## E4 · reward-to-go

一局的奖励序列是 $[r_0, r_1, r_2] = [1, 0, 2]$，$\gamma = 0.9$。reward-to-go 是**从每一步往后**的折扣回报：

$$G_t = r_t + \gamma\, r_{t+1} + \gamma^2 r_{t+2} + \cdots$$

**E4a（纸上）**：算出 $G_0, G_1, G_2$。提示：从最后一步倒着算最省事，$G_t = r_t + \gamma\, G_{t+1}$。

（在这里写：$G_2$ = ？，$G_1$ = ？，$G_0$ = ？）

**E4b（代码）**：写 `reward_to_go(rewards, gamma)`，返回与 `rewards` 等长的 numpy 数组。

In [ ]:
def reward_to_go(rewards, gamma):
    # 在这里写你的代码
    raise NotImplementedError


print(reward_to_go([1.0, 0.0, 2.0], 0.9))
ck.reward_to_go(reward_to_go)

**E4c**：CartPole 每坚持一步得 +1。假设一局坚持了 $T = 200$ 步，画出三种 $\gamma$ 下 $G_t$ 随 $t$ 的变化：

In [ ]:
T = 200
fig, ax = plt.subplots(figsize=(6.5, 3.5))
for gamma in (1.0, 0.99, 0.9):
    ax.plot(reward_to_go(np.ones(T), gamma), label=f"gamma = {gamma}")
ax.set_xlabel("Time step t")
ax.set_ylabel("Reward-to-go G_t")
ax.set_title("gamma = 1 gives a straight ramp T - t; gamma < 1 caps G_t below 1/(1 - gamma)")
ax.legend()
plt.show()

**问题**（每题一两句）：

1. $\gamma = 1$ 时 $G_t$ 是一条什么样的线？写出它的公式（用 $T$ 和 $t$）。
2. $\gamma = 0.9$ 时，开头大部分时间 $G_t$ 都停在 10 附近。$1/(1-\gamma)$ 可以理解成智能体"看得多远"——$\gamma = 0.99$ 时它大约看多少步？为什么图里 $\gamma = 0.99$ 的曲线没有碰到这个上限？
3. （为作业预热）$\gamma = 1$ 时，如果把每一步的 $G_t$ 减去**这一局**所有 $G_t$ 的平均，剩下的东西还和"这一局坚持得久不久"有关吗？

（在这里回答）

## E5 · 随机打破平局的 argmax

讲义 S2 说"Q 表全 0 时 `np.argmax` 永远选向左"。亲眼看一次：在不滑的 FrozenLake 上，用全 0 的 Q 表、`np.argmax` 做贪心策略，跑一局：

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
Q_zero = np.zeros((16, 4))
state, info = env.reset(seed=0)
visited = [state]
while True:
    state, reward, terminated, truncated, info = env.step(int(np.argmax(Q_zero[state])))
    visited.append(state)
    if terminated or truncated:
        break
print(f"走了 {len(visited) - 1} 步 | 到过的状态 {sorted(set(visited))} | terminated={terminated} | truncated={truncated}")

智能体在起点向左撞墙 100 次，直到被时钟叫停（`truncated=True`）。**E5（代码）**：写 `argmax_random(values, rng)`：返回最大值所在的下标；有多个并列最大值时，在它们之中**均匀随机**选一个。提示：`np.flatnonzero(values == values.max())` 给出所有最大值的下标；`rng.choice(候选)` 从中随机抽一个。

In [ ]:
def argmax_random(values, rng):
    # 在这里写你的代码
    raise NotImplementedError


rng = np.random.default_rng(0)
print("argmax_random([0, 0, 0, 0]) 抽 8 次:", [argmax_random(np.zeros(4), rng) for _ in range(8)])
print("argmax_random([1, 3, 3, 0]) 抽 8 次:", [argmax_random(np.array([1.0, 3.0, 3.0, 0.0]), rng) for _ in range(8)])
ck.argmax_random_tiebreak(argmax_random)

同样是全 0 的 Q 表，换成随机打破平局，跑 1000 局看看能不能**碰巧**走到终点：

In [ ]:
rng = np.random.default_rng(0)
wins = 0
for episode in range(1000):
    state, info = env.reset(seed=episode)
    while True:
        state, reward, terminated, truncated, info = env.step(argmax_random(Q_zero[state], rng))
        if terminated or truncated:
            break
    wins += reward
print(f"全 0 的 Q 表 + 随机打破平局：1000 局里到达终点 {int(wins)} 次")

**问题**（每题一两句）：

1. 只要碰到过一次终点，Q-learning 就有了第一个非零的价值、能开始往回传（E3c）。用 `np.argmax` 的智能体为什么可能**永远**等不到这一次（提示：探索率在衰减）？
2. 讲义 S1 的老虎机里用的是另一种写法：`np.argmax(Q + 1e-9 * rng.random(Q.shape))`。它和你的 `argmax_random` 等价吗？有没有它会出错的情况？

**复盘**：今天五个零件里，哪几个是"旧值 + 步长 × (目标 − 旧值)"这个形状？它们的"目标"分别是什么？

（在这里回答）

## E6 · 审计一张 Q 表

讲义 S2 末尾用价值迭代求出了滑冰面的精确最优解，再拿它审计了 5 个 Q-learning 种子。这道题你亲手审计其中一张：下面的 Q 表是讲义 S2 滑冰面、lr = 0.1、**种子 1** 训练出来的（为了不重训，直接给出数值）。它的贪心策略离最优有多远？差在哪几格？

**E6a（代码）**：写 `greedy_as_policy(Q)`：把 Q 表的贪心策略写成一张概率表 `pi`（形状与 `Q` 相同）——每一行里，**并列最大**的动作平分概率，其余为 0。这与讲义 `greedy_policy` 的"随机打破平局"是同一个策略。讲义 S2 里给出过一个逐个状态循环的版本；这里请写成**不用循环**的两行。提示：`Q == Q.max(axis=1, keepdims=True)` 标出每一行最大值的位置。

In [ ]:
def greedy_as_policy(Q):
    # 在这里写你的代码
    raise NotImplementedError


print(greedy_as_policy(np.array([[0.0, 0.0, 0.0, 0.0],
                                 [0.1, 0.5, 0.5, 0.2]])))
ck.greedy_as_policy(greedy_as_policy)

**E6b（给定代码，读懂即可）**：下面几个函数与讲义 S2 相同——`model_arrays` 把 `env.unwrapped.P` 整理成数组 `R`、`C`，`values_by_linear_solve` 用解线性方程的办法（策略迭代）求出 $V^*$ 与最优策略，`success_probability` 精确计算一个策略在 99 步之内到达 G 的概率。运行它，看三件事：这张 Q 表的贪心策略精确成功率是多少；它在哪几格与最优策略不同、每一格单独改对之后成功率回到多少；5 次独立的 1000 回合抽样评估各是多少。

In [ ]:
# 讲义 S2 滑冰面、lr = 0.1、种子 1 训练出的 Q 表（行 = 状态 0–15，列 = 左 / 下 / 右 / 上；保留 6 位小数）
Q_seed1 = np.array([[0.159931, 0.164073, 0.159418, 0.156883],
                    [0.100237, 0.085933, 0.090031, 0.149923],
                    [0.139847, 0.138687, 0.138301, 0.124798],
                    [0.087385, 0.093460, 0.071987, 0.131557],
                    [0.191396, 0.160728, 0.158446, 0.086015],
                    [0.0, 0.0, 0.0, 0.0],
                    [0.108665, 0.115111, 0.106785, 0.034394],
                    [0.0, 0.0, 0.0, 0.0],
                    [0.179661, 0.197817, 0.146795, 0.255716],
                    [0.244926, 0.323939, 0.214406, 0.208054],
                    [0.363469, 0.258163, 0.205342, 0.204562],
                    [0.0, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.0, 0.0],
                    [0.286269, 0.294130, 0.497933, 0.312144],
                    [0.518814, 0.853353, 0.599570, 0.571139],
                    [0.0, 0.0, 0.0, 0.0]])
env_slip = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
P_slip = env_slip.unwrapped.P


def model_arrays(P, n_states, n_actions):
    # 与讲义 S2 相同：R[s, a] = 期望即时奖励；C[s, a, s'] = 到达 s' 且回合没结束的概率
    R, C = np.zeros((n_states, n_actions)), np.zeros((n_states, n_actions, n_states))
    for s in range(n_states):
        for a in range(n_actions):
            for p, s2, r, done in P[s][a]:
                R[s, a] += p * r
                if not done:
                    C[s, a, s2] += p
    return R, C


def values_by_linear_solve(R, C, gamma):
    # 与讲义 S2 相同：策略迭代的矩阵形式——精确评估（解线性方程）与贪心改进交替，返回 (V*, 最优策略, 轮数)
    n_states = R.shape[0]
    states = np.arange(n_states)
    policy = np.zeros(n_states, dtype=int)
    for rounds in range(1, 100):
        V = np.linalg.solve(np.eye(n_states) - gamma * C[states, policy], R[states, policy])
        Q_pi = R + gamma * np.einsum("ijk,k->ij", C, V)
        better = Q_pi[states, policy] < Q_pi.max(axis=1) - 1e-12
        if not better.any():
            return V, policy, rounds
        policy = np.where(better, Q_pi.argmax(axis=1), policy)
    raise RuntimeError("策略迭代没有收敛")


def success_probability(P, policy, horizon):
    # 与讲义 S2 相同：每个起点在 horizon 步之内到达 G 的精确概率（policy[s, a] = 在 s 选 a 的概率）
    n_states, n_actions = policy.shape
    R, C = model_arrays(P, n_states, n_actions)
    S = np.zeros(n_states)
    for _ in range(horizon):
        S = (policy * (R + C @ S)).sum(axis=1)
    return S


def sampled_success(env, pi, n_episodes, rng, max_steps=99, seed=10_000):
    # 与讲义 evaluate_agent 同一口径的抽样评估：按 pi 抽动作、每局最多 99 步，返回到达 G 的回合比例
    wins = 0.0
    for episode in range(n_episodes):
        state, info = env.reset(seed=seed + episode)
        for step in range(max_steps):
            state, reward, terminated, truncated, info = env.step(int(rng.choice(4, p=pi[state])))
            if terminated or truncated:
                break
        wins += reward
    return wins / n_episodes


R_slip, C_slip = model_arrays(P_slip, 16, 4)
V_star, policy_star, _ = values_by_linear_solve(R_slip, C_slip, 0.95)
Q_star = R_slip + 0.95 * C_slip @ V_star
pi_learned = greedy_as_policy(Q_seed1)
exact_learned = success_probability(P_slip, pi_learned, 99)[0]
exact_best = success_probability(P_slip, np.eye(4)[policy_star], 99)[0]
print(f"种子 1 的贪心策略：99 步内成功率 {exact_learned:.4f} | 最优策略 {exact_best:.4f}\n")

ACTION_NAMES = ["左", "下", "右", "上"]
holes_or_goal = np.isin(env_slip.unwrapped.desc.flatten(), [b"H", b"G"])
wrong_states = []
for s in range(16):
    if holes_or_goal[s]:
        continue                                       # 终止格：怎么选都一样
    learned = np.flatnonzero(pi_learned[s] > 0)
    optimal = np.flatnonzero(Q_star[s] >= V_star[s] - 1e-9)
    if not set(learned) <= set(optimal):
        wrong_states.append(s)
        pi_fixed = pi_learned.copy()
        pi_fixed[s] = np.eye(4)[policy_star[s]]        # 只把这一格改成最优动作，其余不动
        print(f"状态 {s:2d}：学到 {[ACTION_NAMES[a] for a in learned]}，最优 {[ACTION_NAMES[a] for a in optimal]} | "
              f"价值损失 V*(s) − Q*(s, 学到的动作) = {V_star[s] - Q_star[s, learned].mean():.4f} | "
              f"只改对这一格，成功率 {exact_learned:.4f} → {success_probability(P_slip, pi_fixed, 99)[0]:.4f}")

rng = np.random.default_rng(1)
samples = [sampled_success(env_slip, pi_learned, 1000, rng, seed=10_000 + 1000 * k) for k in range(5)]
print(f"\n5 次独立的 1000 回合抽样评估：{np.round(samples, 3)}（精确值 {exact_learned:.4f}）")
print("地图：\n" + "\n".join("   " + " ".join(c.decode() for c in row) for row in env_slip.unwrapped.desc))

**问题**（每题两三句，引用上面的数字）：

1. 这张 Q 表只在哪两格与最优策略不同？对照地图（状态按行编号，0–3 是第一行）说说：状态 6 选"下"为什么危险？提示：滑冰面上往一个方向走，会以各 1/3 的概率滑向两侧；状态 6 的左右两边是什么？
2. 状态 0 选错的"价值损失"比状态 6 小得多，可是只改对状态 0，成功率回升得反而更多。为什么？提示：价值损失 $V^*(s) - Q^*(s, a)$ 假设**之后**都按最优策略走；状态 0 选"下"时有 1/3 的概率滑到状态 1——从那里出发，这张 Q 表会把智能体带到哪一格？
3. 5 次 1000 回合的抽样评估彼此差多少？成功率为 $p$ 时，$n$ 个独立回合的抽样估计的标准误约为 $\sqrt{p(1-p)/n}$，这里约多大？如果两个种子的抽样成功率相差 0.02，能据此说谁更好吗？

（在这里回答）

## E7（选做）· 一步 Thompson 采样：臂 0 比臂 1 好的概率

两个臂：臂 0 拉了 4 次，3 次成功、1 次失败；臂 1 也拉了 4 次，1 次成功、3 次失败。先验都是 $\mathrm{Beta}(1, 1)$。

**E7a（纸上）**：写出两个臂的后验。Thompson 采样下一步选臂 0 的概率，为什么恰好等于后验下的 $P(\theta_0 > \theta_1)$？

（在这里写：臂 0 的后验 = ？，臂 1 的后验 = ？；理由：）

**E7b（代码）**：从两个后验里各抽 $10^6$ 个样本，估计 $P(\theta_0 > \theta_1)$。`rng.beta(a, b, n)` 从 $\mathrm{Beta}(a, b)$ 里抽 `n` 个样本。

In [ ]:
rng = np.random.default_rng(0)
n = 1_000_000
# 在这里写你的代码
raise NotImplementedError
print(f"P(θ0 > θ1) ≈ {p_win:.4f}")
ck.beta_win_probability(p_win)

**问题**（每题一两句）：

1. 只看后验均值（臂 0 是 4/6，臂 1 是 2/6）的贪心选择，选臂 0 的概率是多少？Thompson 采样呢？两者之差就是 Thompson 留给臂 1 的"探索"。
2. 讲义 S1 的 `ck.thompson_choose` 用的正是这两个后验（2 万个任务）。为什么可以用你估计的这个数，来判断 `thompson_choose` 写得对不对？
3. 如果两个臂各拉了 40 次（30 成 10 败 vs 10 成 30 败），把参数改一改再跑一次。这个概率怎么变？为什么说 Thompson 的探索会随数据**自己**减少？

（在这里回答）